# Silver Identidade - Benchmark 5 Approaches

## Purpose
Test 5 different strategies for building `silver_identidade` (70M rows, 8 JOINs)
on a memory-constrained machine (16GB RAM, ~3GB free).

## Approaches
1. DuckDB persistent (.duckdb) + loop por UF
2. DuckDB persistent + pre-materialize JOIN
3. Polars lazy + sink_parquet + PartitionBy
4. Polars lazy + loop por UF
5. Polars eager (small tables) + lazy loop (recommended)

## How to use
- Run "Setup" cell first
- Run ONE option at a time (restart kernel between tests for fair comparison)
- Each option cleans the output dir, runs the transformation, and reports metrics
- Compare results in the final cell

## Setup (run before every test)

In [ ]:
import os
import shutil
import time
import tracemalloc
from pathlib import Path
from datetime import datetime

# 1. Definição de Paths
try:
    PROJECT_ROOT = Path(__vsc_ipynb_file__).resolve().parent.parent
except NameError:
    PROJECT_ROOT = Path.cwd().parent

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
SILVER_DIR = PROJECT_ROOT / "data" / "silver"

# --- ALTERAÇÕES PARA ISOLAMENTO DO BENCHMARK ---
# Usamos pastas e arquivos diferentes para não apagar os dados do projeto real (Notebook 14)
OUT_DIR    = SILVER_DIR / "silver_identidade_benchmark" 
DB_FILE    = PROJECT_ROOT / "data" / "bronze_benchmark.duckdb"
# -----------------------------------------------

BD         = BRONZE_DIR.as_posix()

# Limpeza da pasta de benchmark (shutil.rmtree apagará apenas a pasta de teste)
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Utilitário de Monitoramento de Performance
class PerfTracker:
    def __init__(self, name):
        self.name = name
        self.steps = []
        self.t_start = None
        tracemalloc.start()

    def start_step(self, step_name):
        self.current_step = step_name
        self.t_step = time.time()
        self.mem_before = tracemalloc.get_traced_memory()[1] / (1024**2)
        print(f"  >> {step_name}...")

    def end_step(self, rows=0):
        elapsed = time.time() - self.t_step
        mem_after = tracemalloc.get_traced_memory()[1] / (1024**2)
        self.steps.append({
            "step": self.current_step,
            "seconds": round(elapsed, 1),
            "rows": rows,
            "peak_mem_mb": round(mem_after, 1),
        })
        print(f"     {self.current_step}: {elapsed:.1f}s  rows={rows:,}  peak_mem={mem_after:.0f}MB")

    def start(self):
        self.t_start = time.time()
        print(f"{'='*60}\n  BENCHMARK: {self.name}\n{'='*60}")

    def finish(self):
        total = time.time() - self.t_start
        parquet_files = list(OUT_DIR.rglob("*.parquet"))
        total_mb = sum(f.stat().st_size for f in parquet_files) / (1024**2)
        
        total_rows = sum(s["rows"] for s in self.steps if any(k in s["step"].lower() for k in ["loop", "sink", "write", "spark"]))
        if total_rows == 0: total_rows = sum(s["rows"] for s in self.steps)

        tracemalloc.stop()

        print(f"\n{'='*60}\n  RESULTS: {self.name}\n{'='*60}")
        print(f"  Total time      : {total:.1f}s ({total/60:.1f} min)")
        print(f"  Total rows      : {total_rows:,}")
        print(f"  Output size     : {total_mb:,.1f} MB")
        print(f"  Steps breakdown:")
        for s in self.steps:
            print(f"    {s['step']:<40} {s['seconds']:>6.1f}s  rows={s['rows']:>10,}  peak={s['peak_mem_mb']:>8.1f}MB")
        print(f"{'='*60}")
        return {"name": self.name, "total_seconds": round(total,1), "total_rows": total_rows, "output_mb": round(total_mb,1)}

# 3. SQL Fragments (Lógica de Negócio Compartilhada)
# Mantemos aqui para garantir que todos os testes comparem a mesma transformação (Apples to Apples)

SQL_CTE_EST = '''
    SELECT
        cnpj_basico || cnpj_ordem || cnpj_dv   AS cnpj_normalized,
        cnpj_basico, cnpj_ordem, cnpj_dv,
        CASE identificador_matriz_filial
            WHEN 1 THEN 'Matriz' WHEN 2 THEN 'Filial' ELSE 'Desconhecido'
        END AS tipo_estabelecimento,
        nome_fantasia,
        situacao_cadastral AS situacao_cadastral_cod,
        CASE situacao_cadastral
            WHEN '01' THEN 'Nula'             WHEN '02' THEN 'Ativa'
            WHEN '03' THEN 'Suspensa'         WHEN '04' THEN 'Inapta'
            WHEN '05' THEN 'Ativa Nao Regular'
            WHEN '08' THEN 'Baixada'          WHEN '99' THEN 'Exclusao Logica'
            ELSE 'Desconhecida'
        END AS situacao_cadastral_desc,
        CASE WHEN data_situacao_cadastral = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_situacao_cadastral AS VARCHAR), '%Y%m%d')::DATE
        END AS data_situacao_cadastral,
        motivo_situacao_cadastral AS motivo_situacao_cadastral_cod,
        nome_cidade_exterior,
        pais AS pais_cod,
        TRY_STRPTIME(CAST(data_inicio_atividade AS VARCHAR), '%Y%m%d')::DATE AS data_inicio_atividade,
        CASE WHEN cnae_fiscal_principal = '8888888' THEN NULL
             ELSE cnae_fiscal_principal
        END AS cnae_fiscal_principal_cod,
        cnae_fiscal_secundaria,
        tipo_logradouro, logradouro, numero, complemento, bairro, cep,
        municipio AS municipio_cod,
        situacao_especial,
        CASE WHEN data_situacao_especial IS NULL OR data_situacao_especial = 0 THEN NULL
             ELSE TRY_STRPTIME(CAST(data_situacao_especial AS VARCHAR), '%Y%m%d')::DATE
        END AS data_situacao_especial
'''

SQL_CTE_EMP = '''
    SELECT cnpj_basico, razao_social,
        natureza_juridica AS natureza_juridica_cod,
        qualificacao_responsavel AS qualificacao_responsavel_cod,
        TRY_CAST(REPLACE(capital_social, ',', '.') AS DECIMAL(15,2)) AS capital_social,
        CASE porte_empresa
            WHEN '01' THEN 'Micro Empresa'
            WHEN '03' THEN 'Empresa de Pequeno Porte'
            WHEN '05' THEN 'Demais' ELSE NULL
        END AS porte_empresa,
        ente_federativo_responsavel
'''

SQL_CTE_SIM = '''
    SELECT cnpj_basico,
        CASE opcao_simples WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END AS opcao_simples,
        CASE opcao_mei WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE ELSE NULL END AS opcao_mei,
        TRY_STRPTIME(CAST(data_opcao_simples AS VARCHAR), '%Y%m%d')::DATE AS data_opcao_simples,
        CASE WHEN data_exclusao_simples = '00000000' THEN NULL
             ELSE TRY_STRPTIME(data_exclusao_simples, '%Y%m%d')::DATE END AS data_exclusao_simples,
        CASE WHEN data_opcao_mei = '00000000' THEN NULL
             ELSE TRY_STRPTIME(data_opcao_mei, '%Y%m%d')::DATE END AS data_opcao_mei,
        CASE WHEN data_exclusao_mei = '00000000' THEN NULL
             ELSE TRY_STRPTIME(data_exclusao_mei, '%Y%m%d')::DATE END AS data_exclusao_mei
'''

SQL_SELECT_FINAL = '''
    SELECT
        est.cnpj_normalized, est.cnpj_basico, est.cnpj_ordem, est.cnpj_dv,
        est.tipo_estabelecimento, est.nome_fantasia,
        est.situacao_cadastral_cod, est.situacao_cadastral_desc,
        est.data_situacao_cadastral,
        est.motivo_situacao_cadastral_cod,
        mot.motivo_descricao AS motivo_situacao_cadastral_desc,
        est.data_inicio_atividade,
        est.cnae_fiscal_principal_cod,
        cnae.cnae_descricao AS cnae_fiscal_principal_desc,
        est.cnae_fiscal_secundaria,
        est.tipo_logradouro, est.logradouro, est.numero, est.complemento,
        est.bairro, est.cep,
        est.municipio_cod,
        mun.municipio_descricao AS municipio_desc,
        est.nome_cidade_exterior,
        est.pais_cod,
        pais.pais_descricao AS pais_desc,
        est.situacao_especial, est.data_situacao_especial,
        emp.razao_social,
        emp.natureza_juridica_cod,
        nat.natureza_descricao AS natureza_juridica_desc,
        emp.qualificacao_responsavel_cod,
        COALESCE(qual.qualificacao_descricao, 'Nao encontrada na tabela de dominio') AS qualificacao_responsavel_desc,
        emp.capital_social, emp.porte_empresa, emp.ente_federativo_responsavel,
        sim.opcao_simples, sim.data_opcao_simples, sim.data_exclusao_simples,
        sim.opcao_mei, sim.data_opcao_mei, sim.data_exclusao_mei,
        CURRENT_TIMESTAMP AS _silver_loaded_at
'''

SQL_JOINS = '''
    FROM cte_est est
    LEFT JOIN cte_emp emp ON est.cnpj_basico = emp.cnpj_basico
    LEFT JOIN cte_sim sim ON est.cnpj_basico = sim.cnpj_basico
    LEFT JOIN dom_naturezas nat ON emp.natureza_juridica_cod = nat.natureza_codigo
    LEFT JOIN dom_qualificacoes qual ON emp.qualificacao_responsavel_cod = qual.qualificacao_codigo
    LEFT JOIN dom_cnaes cnae ON est.cnae_fiscal_principal_cod = cnae.cnae_codigo
    LEFT JOIN dom_municipios mun ON est.municipio_cod = mun.municipio_codigo
    LEFT JOIN dom_paises pais ON est.pais_cod = pais.pais_codigo
    LEFT JOIN dom_motivos mot ON est.motivo_situacao_cadastral_cod = mot.motivo_codigo
'''

# 4. Configurações Polars (Colunas Necessárias)
EST_COLS_NEEDED = ["cnpj_basico", "cnpj_ordem", "cnpj_dv", "identificador_matriz_filial", "nome_fantasia", "situacao_cadastral", "data_situacao_cadastral", "motivo_situacao_cadastral", "nome_cidade_exterior", "pais", "data_inicio_atividade", "cnae_fiscal_principal", "cnae_fiscal_secundaria", "tipo_logradouro", "logradouro", "numero", "complemento", "bairro", "cep", "uf", "municipio", "situacao_especial", "data_situacao_especial"]
EMP_COLS_NEEDED = ["cnpj_basico", "razao_social", "natureza_juridica", "qualificacao_responsavel", "capital_social", "porte_empresa", "ente_federativo_responsavel"]
SIM_COLS_NEEDED = ["cnpj_basico", "opcao_simples", "opcao_mei", "data_opcao_simples", "data_exclusao_simples", "data_opcao_mei", "data_exclusao_mei"]

def duckdb_register_domains(con):
    domains = {"dom_naturezas": "rf_naturezas", "dom_qualificacoes": "rf_qualificacoes", "dom_cnaes": "rf_cnaes", "dom_municipios": "rf_municipios", "dom_paises": "rf_paises", "dom_motivos": "rf_motivos"}
    for view, folder in domains.items():
        con.execute(f"CREATE OR REPLACE VIEW {view} AS SELECT * FROM read_parquet('{BD}/{folder}/**/*.parquet')")

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"BRONZE_DIR:   {BRONZE_DIR}")
print(f"OUTPUT_DIR:   {OUT_DIR}")
print("\nSetup ready. Lógica de SQL carregada e diretórios de benchmark isolados.")

## Option 1 - DuckDB Persistent (.duckdb) + Loop por UF

**Strategy:** Materialize 3 Bronze tables into a `.duckdb` file, then loop by UF.
DuckDB uses row group statistics to skip irrelevant data for each UF.

**Expected:** ~30 min | **Success probability:** 70%
**Risk:** Hash table of rf_empresas (67M) rebuilt each iteration

In [ ]:
import duckdb

perf = PerfTracker("Option 1: DuckDB persistent + loop UF")
perf.start()

# Clean
if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if DB_FILE.exists(): DB_FILE.unlink()

con = duckdb.connect(str(DB_FILE))
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '4GB'")
con.execute("SET preserve_insertion_order = false")

# Step 1: Materialize Bronze
perf.start_step("Materialize rf_estabelecimentos")
con.execute(f"CREATE TABLE rf_estabelecimentos AS SELECT * FROM read_parquet('{BD}/rf_estabelecimentos/**/*.parquet')")
n = con.execute("SELECT count(*) FROM rf_estabelecimentos").fetchone()[0]
perf.end_step(n)

perf.start_step("Materialize rf_empresas")
con.execute(f"CREATE TABLE rf_empresas AS SELECT * FROM read_parquet('{BD}/rf_empresas/**/*.parquet')")
n = con.execute("SELECT count(*) FROM rf_empresas").fetchone()[0]
perf.end_step(n)

perf.start_step("Materialize rf_simples")
con.execute(f"CREATE TABLE rf_simples AS SELECT * FROM read_parquet('{BD}/rf_simples/**/*.parquet')")
n = con.execute("SELECT count(*) FROM rf_simples").fetchone()[0]
perf.end_step(n)

duckdb_register_domains(con)

# Step 2: Loop by UF
ufs = con.execute("SELECT DISTINCT uf FROM rf_estabelecimentos WHERE uf IS NOT NULL ORDER BY uf").fetchall()
ufs = [r[0] for r in ufs]

perf.start_step(f"Loop {len(ufs)} UFs (join + write)")
grand_total = 0
for uf in ufs:
    out_p = OUT_DIR / f"uf={uf}"
    out_p.mkdir(parents=True, exist_ok=True)
    out_f = (out_p / "data.parquet").as_posix()
    con.execute(f'''
        COPY (
            WITH
                cte_est AS ({SQL_CTE_EST} FROM rf_estabelecimentos WHERE uf = '{uf}'),
                cte_emp AS ({SQL_CTE_EMP} FROM rf_empresas),
                cte_sim AS ({SQL_CTE_SIM} FROM rf_simples)
            {SQL_SELECT_FINAL}, '{uf}' AS uf
            {SQL_JOINS}
        ) TO '{out_f}' (FORMAT PARQUET)
    ''')
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out_f}')").fetchone()[0]
    grand_total += n
    print(f"     uf={uf:<3} rows={n:>10,}")
perf.end_step(grand_total)

con.close()
result_1 = perf.finish()

# Cleanup .duckdb
if DB_FILE.exists():
    db_size = DB_FILE.stat().st_size / (1024**3)
    print(f"  bronze.duckdb size: {db_size:.1f} GB (can be deleted)")

## Option 2 - DuckDB Persistent + Pre-materialize JOIN

**Strategy:** Materialize the full joined result inside `.duckdb`, then loop by UF just COPYing to Parquet.

**Expected:** ~45 min | **Success probability:** 60%
**Risk:** The full join inside .duckdb may spill heavily

In [ ]:
import duckdb

perf = PerfTracker("Option 2: DuckDB persistent + pre-join")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if DB_FILE.exists(): DB_FILE.unlink()

con = duckdb.connect(str(DB_FILE))
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '4GB'")
con.execute("SET preserve_insertion_order = false")

# Step 1: Materialize the full join
duckdb_register_domains(con)

perf.start_step("Materialize full join into silver_joined")
con.execute(f'''
    CREATE TABLE silver_joined AS
    WITH
        cte_est AS ({SQL_CTE_EST} FROM read_parquet('{BD}/rf_estabelecimentos/**/*.parquet')),
        cte_emp AS ({SQL_CTE_EMP} FROM read_parquet('{BD}/rf_empresas/**/*.parquet')),
        cte_sim AS ({SQL_CTE_SIM} FROM read_parquet('{BD}/rf_simples/**/*.parquet'))
    {SQL_SELECT_FINAL}, est.uf AS uf  -- use source uf, add missing column
    {SQL_JOINS.replace("est.municipio_cod", "est.municipio_cod")}
''')
n = con.execute("SELECT count(*) FROM silver_joined").fetchone()[0]
perf.end_step(n)

# Step 2: Loop by UF - just COPY, no join
ufs = con.execute("SELECT DISTINCT uf FROM silver_joined WHERE uf IS NOT NULL ORDER BY uf").fetchall()
ufs = [r[0] for r in ufs]

perf.start_step(f"Write {len(ufs)} UF partitions")
grand_total = 0
for uf in ufs:
    out_p = OUT_DIR / f"uf={uf}"
    out_p.mkdir(parents=True, exist_ok=True)
    out_f = (out_p / "data.parquet").as_posix()
    con.execute(f"COPY (SELECT * FROM silver_joined WHERE uf = '{uf}') TO '{out_f}' (FORMAT PARQUET)")
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out_f}')").fetchone()[0]
    grand_total += n
    print(f"     uf={uf:<3} rows={n:>10,}")
perf.end_step(grand_total)

con.close()
result_2 = perf.finish()

if DB_FILE.exists():
    db_size = DB_FILE.stat().st_size / (1024**3)
    print(f"  bronze.duckdb size: {db_size:.1f} GB (can be deleted)")

## Option 3 - Polars Lazy + sink_parquet + PartitionBy

**Strategy:** Build the entire pipeline as a single Polars lazy query.
Use `sink_parquet` with `PartitionBy` to stream results directly to disk.

**Expected:** ~20 min | **Success probability:** 85%
**Risk:** Streaming engine may not support all operations; falls back to eager

In [ ]:
import polars as pl

perf = PerfTracker("Option 3: Polars sink_parquet")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Step 1: Load domain tables (small, eager)
perf.start_step("Load domain tables")
dom_nat  = pl.read_parquet(f"{BD}/rf_naturezas/**/*.parquet").rename({"natureza_codigo":"_nat_cod","natureza_descricao":"natureza_juridica_desc"})
dom_qual = pl.read_parquet(f"{BD}/rf_qualificacoes/**/*.parquet").rename({"qualificacao_codigo":"_qual_cod","qualificacao_descricao":"qualificacao_responsavel_desc"})
dom_cnae = pl.read_parquet(f"{BD}/rf_cnaes/**/*.parquet").rename({"cnae_codigo":"_cnae_cod","cnae_descricao":"cnae_fiscal_principal_desc"})
dom_mun  = pl.read_parquet(f"{BD}/rf_municipios/**/*.parquet").rename({"municipio_codigo":"_mun_cod","municipio_descricao":"municipio_desc"})
dom_pais = pl.read_parquet(f"{BD}/rf_paises/**/*.parquet").rename({"pais_codigo":"_pais_cod","pais_descricao":"pais_desc"})
dom_mot  = pl.read_parquet(f"{BD}/rf_motivos/**/*.parquet").rename({"motivo_codigo":"_mot_cod","motivo_descricao":"motivo_situacao_cadastral_desc"})
perf.end_step(0)

# Step 2: Build lazy pipeline
perf.start_step("Build lazy pipeline + sink_parquet")

sit_map = {"01":"Nula","02":"Ativa","03":"Suspensa","04":"Inapta","05":"Ativa Nao Regular","08":"Baixada","99":"Exclusao Logica"}
porte_map = {"01":"Micro Empresa","03":"Empresa de Pequeno Porte","05":"Demais"}

lf_est = (
    pl.scan_parquet(f"{BD}/rf_estabelecimentos/**/*.parquet", hive_partitioning=False)
    .select(EST_COLS_NEEDED)
    .with_columns([
        (pl.col("cnpj_basico") + pl.col("cnpj_ordem") + pl.col("cnpj_dv")).alias("cnpj_normalized"),
        pl.when(pl.col("identificador_matriz_filial")==1).then(pl.lit("Matriz"))
          .when(pl.col("identificador_matriz_filial")==2).then(pl.lit("Filial"))
          .otherwise(pl.lit("Desconhecido")).alias("tipo_estabelecimento"),
        pl.col("situacao_cadastral").replace_strict(sit_map, default="Desconhecida").alias("situacao_cadastral_desc"),
        pl.when(pl.col("data_situacao_cadastral")==0).then(None)
          .otherwise(pl.col("data_situacao_cadastral").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
          .alias("data_situacao_cadastral_dt"),
        pl.col("data_inicio_atividade").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False).alias("data_inicio_atividade_dt"),
        pl.when(pl.col("cnae_fiscal_principal")=="8888888").then(None)
          .otherwise(pl.col("cnae_fiscal_principal")).alias("cnae_fiscal_principal_clean"),
        pl.when(pl.col("data_situacao_especial").is_null() | (pl.col("data_situacao_especial")==0)).then(None)
          .otherwise(pl.col("data_situacao_especial").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
          .alias("data_situacao_especial_dt"),
    ])
    .drop(["identificador_matriz_filial","data_situacao_cadastral","data_inicio_atividade","cnae_fiscal_principal","data_situacao_especial"])
    .rename({"situacao_cadastral":"situacao_cadastral_cod","motivo_situacao_cadastral":"motivo_situacao_cadastral_cod",
             "pais":"pais_cod","municipio":"municipio_cod",
             "data_situacao_cadastral_dt":"data_situacao_cadastral","data_inicio_atividade_dt":"data_inicio_atividade",
             "cnae_fiscal_principal_clean":"cnae_fiscal_principal_cod","data_situacao_especial_dt":"data_situacao_especial"})
)

lf_emp = (
    pl.scan_parquet(f"{BD}/rf_empresas/**/*.parquet", hive_partitioning=False)
    .select(EMP_COLS_NEEDED)
    .with_columns([
        pl.col("capital_social").str.replace(",",".").cast(pl.Float64, strict=False).alias("capital_social_num"),
        pl.col("porte_empresa").replace_strict(porte_map, default=None).alias("porte_empresa_desc"),
    ])
    .drop(["capital_social","porte_empresa"])
    .rename({"natureza_juridica":"natureza_juridica_cod","qualificacao_responsavel":"qualificacao_responsavel_cod",
             "capital_social_num":"capital_social","porte_empresa_desc":"porte_empresa"})
)

lf_sim = (
    pl.scan_parquet(f"{BD}/rf_simples/**/*.parquet", hive_partitioning=False)
    .select(SIM_COLS_NEEDED)
    .with_columns([
        pl.col("opcao_simples").map_elements(lambda x: True if x=="S" else (False if x=="N" else None), return_dtype=pl.Boolean).alias("opcao_simples_bool"),
        pl.col("opcao_mei").map_elements(lambda x: True if x=="S" else (False if x=="N" else None), return_dtype=pl.Boolean).alias("opcao_mei_bool"),
        pl.col("data_opcao_simples").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False),
        pl.when(pl.col("data_exclusao_simples")=="00000000").then(None)
          .otherwise(pl.col("data_exclusao_simples").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_simples_dt"),
        pl.when(pl.col("data_opcao_mei")=="00000000").then(None)
          .otherwise(pl.col("data_opcao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_opcao_mei_dt"),
        pl.when(pl.col("data_exclusao_mei")=="00000000").then(None)
          .otherwise(pl.col("data_exclusao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_mei_dt"),
    ])
    .drop(["opcao_simples","opcao_mei","data_exclusao_simples","data_opcao_mei","data_exclusao_mei"])
    .rename({"opcao_simples_bool":"opcao_simples","opcao_mei_bool":"opcao_mei",
             "data_exclusao_simples_dt":"data_exclusao_simples","data_opcao_mei_dt":"data_opcao_mei","data_exclusao_mei_dt":"data_exclusao_mei"})
)

# Join pipeline
result = (
    lf_est
    .join(lf_emp, on="cnpj_basico", how="left")
    .join(lf_sim, on="cnpj_basico", how="left")
    .join(dom_nat.lazy(), left_on="natureza_juridica_cod", right_on="_nat_cod", how="left")
    .join(dom_qual.lazy(), left_on="qualificacao_responsavel_cod", right_on="_qual_cod", how="left")
    .join(dom_cnae.lazy(), left_on="cnae_fiscal_principal_cod", right_on="_cnae_cod", how="left")
    .join(dom_mun.lazy(), left_on="municipio_cod", right_on="_mun_cod", how="left")
    .join(dom_pais.lazy(), left_on="pais_cod", right_on="_pais_cod", how="left")
    .join(dom_mot.lazy(), left_on="motivo_situacao_cadastral_cod", right_on="_mot_cod", how="left")
    .with_columns(pl.lit(datetime.now()).alias("_silver_loaded_at"))
)

try:
    result.sink_parquet(
        pl.PartitionBy(str(OUT_DIR), key="uf"),
        mkdir=True
    )
    # Count output
    total_rows = 0
    for d in sorted(OUT_DIR.iterdir()):
        if d.is_dir() and d.name.startswith("uf="):
            for f in d.glob("*.parquet"):
                n = pl.scan_parquet(str(f)).select(pl.len()).collect().item()
                total_rows += n
    perf.end_step(total_rows)
except Exception as e:
    print(f"  sink_parquet failed: {e}")
    print("  Falling back to collect(streaming=True) per UF...")
    perf.end_step(0)

result_3 = perf.finish()

## Option 4 - Polars Lazy + Loop por UF

**Strategy:** Same lazy Polars pipeline, but loop by UF manually.
Each iteration filters rf_estabelecimentos by UF before joining.

**Expected:** ~30 min | **Success probability:** 90%
**Risk:** Hash table of rf_empresas (67M) rebuilt each iteration, but Polars handles it better

In [ ]:
import polars as pl

perf = PerfTracker("Option 4: Polars lazy + loop UF")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Domain tables (small, eager)
perf.start_step("Load domain tables")
dom_nat  = pl.read_parquet(f"{BD}/rf_naturezas/**/*.parquet").rename({"natureza_codigo":"_nat_cod","natureza_descricao":"natureza_juridica_desc"})
dom_qual = pl.read_parquet(f"{BD}/rf_qualificacoes/**/*.parquet").rename({"qualificacao_codigo":"_qual_cod","qualificacao_descricao":"qualificacao_responsavel_desc"})
dom_cnae = pl.read_parquet(f"{BD}/rf_cnaes/**/*.parquet").rename({"cnae_codigo":"_cnae_cod","cnae_descricao":"cnae_fiscal_principal_desc"})
dom_mun  = pl.read_parquet(f"{BD}/rf_municipios/**/*.parquet").rename({"municipio_codigo":"_mun_cod","municipio_descricao":"municipio_desc"})
dom_pais = pl.read_parquet(f"{BD}/rf_paises/**/*.parquet").rename({"pais_codigo":"_pais_cod","pais_descricao":"pais_desc"})
dom_mot  = pl.read_parquet(f"{BD}/rf_motivos/**/*.parquet").rename({"motivo_codigo":"_mot_cod","motivo_descricao":"motivo_situacao_cadastral_desc"})
perf.end_step(0)

# Get UF list
ufs = pl.read_parquet(f"{BD}/rf_estabelecimentos/**/*.parquet", columns=["uf"]).get_column("uf").unique().sort().to_list()
ufs = [u for u in ufs if u is not None]

sit_map = {"01":"Nula","02":"Ativa","03":"Suspensa","04":"Inapta","05":"Ativa Nao Regular","08":"Baixada","99":"Exclusao Logica"}
porte_map = {"01":"Micro Empresa","03":"Empresa de Pequeno Porte","05":"Demais"}

perf.start_step(f"Loop {len(ufs)} UFs (lazy join + collect + write)")
grand_total = 0
for uf in ufs:
    t0 = time.time()
    lf_est = (
        pl.scan_parquet(f"{BD}/rf_estabelecimentos/**/*.parquet", hive_partitioning=False)
        .filter(pl.col("uf") == uf)
        .select(EST_COLS_NEEDED)
        .with_columns([
            (pl.col("cnpj_basico") + pl.col("cnpj_ordem") + pl.col("cnpj_dv")).alias("cnpj_normalized"),
            pl.when(pl.col("identificador_matriz_filial")==1).then(pl.lit("Matriz"))
              .when(pl.col("identificador_matriz_filial")==2).then(pl.lit("Filial"))
              .otherwise(pl.lit("Desconhecido")).alias("tipo_estabelecimento"),
            pl.col("situacao_cadastral").replace_strict(sit_map, default="Desconhecida").alias("situacao_cadastral_desc"),
            pl.when(pl.col("data_situacao_cadastral")==0).then(None)
              .otherwise(pl.col("data_situacao_cadastral").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
              .alias("data_situacao_cadastral_dt"),
            pl.col("data_inicio_atividade").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False).alias("data_inicio_atividade_dt"),
            pl.when(pl.col("cnae_fiscal_principal")=="8888888").then(None)
              .otherwise(pl.col("cnae_fiscal_principal")).alias("cnae_fiscal_principal_clean"),
            pl.when(pl.col("data_situacao_especial").is_null() | (pl.col("data_situacao_especial")==0)).then(None)
              .otherwise(pl.col("data_situacao_especial").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
              .alias("data_situacao_especial_dt"),
        ])
        .drop(["identificador_matriz_filial","data_situacao_cadastral","data_inicio_atividade","cnae_fiscal_principal","data_situacao_especial"])
        .rename({"situacao_cadastral":"situacao_cadastral_cod","motivo_situacao_cadastral":"motivo_situacao_cadastral_cod",
                 "pais":"pais_cod","municipio":"municipio_cod",
                 "data_situacao_cadastral_dt":"data_situacao_cadastral","data_inicio_atividade_dt":"data_inicio_atividade",
                 "cnae_fiscal_principal_clean":"cnae_fiscal_principal_cod","data_situacao_especial_dt":"data_situacao_especial"})
    )
    lf_emp = (
        pl.scan_parquet(f"{BD}/rf_empresas/**/*.parquet", hive_partitioning=False)
        .select(EMP_COLS_NEEDED)
        .with_columns([
            pl.col("capital_social").str.replace(",",".").cast(pl.Float64, strict=False).alias("capital_social_num"),
            pl.col("porte_empresa").replace_strict(porte_map, default=None).alias("porte_empresa_desc"),
        ])
        .drop(["capital_social","porte_empresa"])
        .rename({"natureza_juridica":"natureza_juridica_cod","qualificacao_responsavel":"qualificacao_responsavel_cod",
                 "capital_social_num":"capital_social","porte_empresa_desc":"porte_empresa"})
    )
    lf_sim = (
        pl.scan_parquet(f"{BD}/rf_simples/**/*.parquet", hive_partitioning=False)
        .select(SIM_COLS_NEEDED)
        .with_columns([
            (pl.col("opcao_simples") == "S").alias("opcao_simples_bool"),
            (pl.col("opcao_mei") == "S").alias("opcao_mei_bool"),
            pl.col("data_opcao_simples").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False),
            pl.when(pl.col("data_exclusao_simples")=="00000000").then(None)
              .otherwise(pl.col("data_exclusao_simples").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_simples_dt"),
            pl.when(pl.col("data_opcao_mei")=="00000000").then(None)
              .otherwise(pl.col("data_opcao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_opcao_mei_dt"),
            pl.when(pl.col("data_exclusao_mei")=="00000000").then(None)
              .otherwise(pl.col("data_exclusao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_mei_dt"),
        ])
        .drop(["opcao_simples","opcao_mei","data_exclusao_simples","data_opcao_mei","data_exclusao_mei"])
        .rename({"opcao_simples_bool":"opcao_simples","opcao_mei_bool":"opcao_mei",
                 "data_exclusao_simples_dt":"data_exclusao_simples","data_opcao_mei_dt":"data_opcao_mei","data_exclusao_mei_dt":"data_exclusao_mei"})
    )
    try:
        df = (
            lf_est
            .join(lf_emp, on="cnpj_basico", how="left")
            .join(lf_sim, on="cnpj_basico", how="left")
            .join(dom_nat.lazy(), left_on="natureza_juridica_cod", right_on="_nat_cod", how="left")
            .join(dom_qual.lazy(), left_on="qualificacao_responsavel_cod", right_on="_qual_cod", how="left")
            .join(dom_cnae.lazy(), left_on="cnae_fiscal_principal_cod", right_on="_cnae_cod", how="left")
            .join(dom_mun.lazy(), left_on="municipio_cod", right_on="_mun_cod", how="left")
            .join(dom_pais.lazy(), left_on="pais_cod", right_on="_pais_cod", how="left")
            .join(dom_mot.lazy(), left_on="motivo_situacao_cadastral_cod", right_on="_mot_cod", how="left")
            .with_columns(pl.lit(datetime.now()).alias("_silver_loaded_at"))
            .collect(streaming=True)
        )
        out_p = OUT_DIR / f"uf={uf}"
        out_p.mkdir(parents=True, exist_ok=True)
        df.write_parquet(out_p / "data.parquet")
        n = len(df)
        grand_total += n
        del df
        print(f"     uf={uf:<3} rows={n:>10,}  {time.time()-t0:.1f}s  cumulative={grand_total:,}")
    except Exception as e:
        print(f"     uf={uf:<3} ERROR: {str(e)[:60]}")

perf.end_step(grand_total)
result_4 = perf.finish()

## Option 5 - Polars Eager (small tables) + Lazy Loop (RECOMMENDED)

**Strategy:** Load rf_empresas and rf_simples eagerly into memory (~5GB).
Loop by UF with lazy scan of rf_estabelecimentos. Join against pre-loaded DataFrames.

**Expected:** ~20 min | **Success probability:** 95%
**Key advantage:** Hash table built ONCE, reused 28 times

In [ ]:
import polars as pl

perf = PerfTracker("Option 5: Polars eager small + lazy loop")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

sit_map = {"01":"Nula","02":"Ativa","03":"Suspensa","04":"Inapta","05":"Ativa Nao Regular","08":"Baixada","99":"Exclusao Logica"}
porte_map = {"01":"Micro Empresa","03":"Empresa de Pequeno Porte","05":"Demais"}

# Step 1: Load small tables eagerly (only needed columns)
perf.start_step("Load rf_empresas eagerly")
df_emp = (
    pl.read_parquet(f"{BD}/rf_empresas/**/*.parquet", columns=EMP_COLS_NEEDED, hive_partitioning=False)
    .with_columns([
        pl.col("capital_social").str.replace(",",".").cast(pl.Float64, strict=False).alias("capital_social_num"),
        pl.col("porte_empresa").replace_strict(porte_map, default=None).alias("porte_empresa_desc"),
    ])
    .drop(["capital_social","porte_empresa"])
    .rename({"natureza_juridica":"natureza_juridica_cod","qualificacao_responsavel":"qualificacao_responsavel_cod",
             "capital_social_num":"capital_social","porte_empresa_desc":"porte_empresa"})
)
perf.end_step(len(df_emp))
print(f"     Memory: {df_emp.estimated_size('mb'):.0f} MB")

perf.start_step("Load rf_simples eagerly")
df_sim = (
    pl.read_parquet(f"{BD}/rf_simples/**/*.parquet", columns=SIM_COLS_NEEDED, hive_partitioning=False)
    .with_columns([
        (pl.col("opcao_simples") == "S").alias("opcao_simples_bool"),
        (pl.col("opcao_mei") == "S").alias("opcao_mei_bool"),
        pl.col("data_opcao_simples").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False),
        pl.when(pl.col("data_exclusao_simples")=="00000000").then(None)
          .otherwise(pl.col("data_exclusao_simples").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_simples_dt"),
        pl.when(pl.col("data_opcao_mei")=="00000000").then(None)
          .otherwise(pl.col("data_opcao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_opcao_mei_dt"),
        pl.when(pl.col("data_exclusao_mei")=="00000000").then(None)
          .otherwise(pl.col("data_exclusao_mei").str.strptime(pl.Date, "%Y%m%d", strict=False)).alias("data_exclusao_mei_dt"),
    ])
    .drop(["opcao_simples","opcao_mei","data_exclusao_simples","data_opcao_mei","data_exclusao_mei"])
    .rename({"opcao_simples_bool":"opcao_simples","opcao_mei_bool":"opcao_mei",
             "data_exclusao_simples_dt":"data_exclusao_simples","data_opcao_mei_dt":"data_opcao_mei","data_exclusao_mei_dt":"data_exclusao_mei"})
)
perf.end_step(len(df_sim))
print(f"     Memory: {df_sim.estimated_size('mb'):.0f} MB")

# Step 2: Load domain tables
perf.start_step("Load domain tables")
dom_nat  = pl.read_parquet(f"{BD}/rf_naturezas/**/*.parquet").rename({"natureza_codigo":"_nat_cod","natureza_descricao":"natureza_juridica_desc"})
dom_qual = pl.read_parquet(f"{BD}/rf_qualificacoes/**/*.parquet").rename({"qualificacao_codigo":"_qual_cod","qualificacao_descricao":"qualificacao_responsavel_desc"})
dom_cnae = pl.read_parquet(f"{BD}/rf_cnaes/**/*.parquet").rename({"cnae_codigo":"_cnae_cod","cnae_descricao":"cnae_fiscal_principal_desc"})
dom_mun  = pl.read_parquet(f"{BD}/rf_municipios/**/*.parquet").rename({"municipio_codigo":"_mun_cod","municipio_descricao":"municipio_desc"})
dom_pais = pl.read_parquet(f"{BD}/rf_paises/**/*.parquet").rename({"pais_codigo":"_pais_cod","pais_descricao":"pais_desc"})
dom_mot  = pl.read_parquet(f"{BD}/rf_motivos/**/*.parquet").rename({"motivo_codigo":"_mot_cod","motivo_descricao":"motivo_situacao_cadastral_desc"})
perf.end_step(0)

# Step 3: Get UF list efficiently
perf.start_step("Get UF list")
ufs = (
    pl.scan_parquet(f"{BD}/rf_estabelecimentos/**/*.parquet", hive_partitioning=False)
    .select("uf").unique().collect().get_column("uf").sort().to_list()
)
ufs = [u for u in ufs if u is not None]
perf.end_step(len(ufs))

# Step 4: Loop by UF
perf.start_step(f"Loop {len(ufs)} UFs (lazy scan + join preloaded + write)")
grand_total = 0
for uf in ufs:
    t0 = time.time()
    try:
        lf_est = (
            pl.scan_parquet(f"{BD}/rf_estabelecimentos/**/*.parquet", hive_partitioning=False)
            .filter(pl.col("uf") == uf)
            .select(EST_COLS_NEEDED)
            .with_columns([
                (pl.col("cnpj_basico") + pl.col("cnpj_ordem") + pl.col("cnpj_dv")).alias("cnpj_normalized"),
                pl.when(pl.col("identificador_matriz_filial")==1).then(pl.lit("Matriz"))
                  .when(pl.col("identificador_matriz_filial")==2).then(pl.lit("Filial"))
                  .otherwise(pl.lit("Desconhecido")).alias("tipo_estabelecimento"),
                pl.col("situacao_cadastral").replace_strict(sit_map, default="Desconhecida").alias("situacao_cadastral_desc"),
                pl.when(pl.col("data_situacao_cadastral")==0).then(None)
                  .otherwise(pl.col("data_situacao_cadastral").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
                  .alias("data_situacao_cadastral_dt"),
                pl.col("data_inicio_atividade").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False).alias("data_inicio_atividade_dt"),
                pl.when(pl.col("cnae_fiscal_principal")=="8888888").then(None)
                  .otherwise(pl.col("cnae_fiscal_principal")).alias("cnae_fiscal_principal_clean"),
                pl.when(pl.col("data_situacao_especial").is_null() | (pl.col("data_situacao_especial")==0)).then(None)
                  .otherwise(pl.col("data_situacao_especial").cast(pl.String).str.strptime(pl.Date, "%Y%m%d", strict=False))
                  .alias("data_situacao_especial_dt"),
            ])
            .drop(["identificador_matriz_filial","data_situacao_cadastral","data_inicio_atividade","cnae_fiscal_principal","data_situacao_especial"])
            .rename({"situacao_cadastral":"situacao_cadastral_cod","motivo_situacao_cadastral":"motivo_situacao_cadastral_cod",
                     "pais":"pais_cod","municipio":"municipio_cod",
                     "data_situacao_cadastral_dt":"data_situacao_cadastral","data_inicio_atividade_dt":"data_inicio_atividade",
                     "cnae_fiscal_principal_clean":"cnpj_fiscal_principal_cod","data_situacao_especial_dt":"data_situacao_especial"})
        )

        df = (
            lf_est
            .join(df_emp.lazy(), on="cnpj_basico", how="left")
            .join(df_sim.lazy(), on="cnpj_basico", how="left")
            .join(dom_nat.lazy(), left_on="natureza_juridica_cod", right_on="_nat_cod", how="left")
            .join(dom_qual.lazy(), left_on="qualificacao_responsavel_cod", right_on="_qual_cod", how="left")
            .join(dom_cnae.lazy(), left_on="cnpj_fiscal_principal_cod", right_on="_cnae_cod", how="left")
            .join(dom_mun.lazy(), left_on="municipio_cod", right_on="_mun_cod", how="left")
            .join(dom_pais.lazy(), left_on="pais_cod", right_on="_pais_cod", how="left")
            .join(dom_mot.lazy(), left_on="motivo_situacao_cadastral_cod", right_on="_mot_cod", how="left")
            .with_columns(pl.lit(datetime.now()).alias("_silver_loaded_at"))
            .collect(streaming=True)
        )

        out_p = OUT_DIR / f"uf={uf}"
        out_p.mkdir(parents=True, exist_ok=True)
        df.write_parquet(out_p / "data.parquet")
        n = len(df)
        grand_total += n
        del df
        print(f"     uf={uf:<3} rows={n:>10,}  {time.time()-t0:.1f}s  cumulative={grand_total:,}")
    except Exception as e:
        print(f"     uf={uf:<3} ERROR: {str(e)[:80]}")

perf.end_step(grand_total)
result_5 = perf.finish()

### Opção 6 - DuckDB Pré-Ordenado

In [ ]:
import duckdb

perf = PerfTracker("Option 6: DuckDB pre-sorted (Sort-Merge)")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if DB_FILE.exists(): DB_FILE.unlink()

con = duckdb.connect(str(DB_FILE))
con.execute("SET threads TO 4")
con.execute("SET memory_limit = '4GB'")

# A grande diferença: forçamos a ordenação na criação da tabela
perf.start_step("Materialize sorted rf_empresas")
con.execute(f"CREATE TABLE rf_empresas_sorted AS SELECT * FROM read_parquet('{BD}/rf_empresas/**/*.parquet') ORDER BY cnpj_basico")
perf.end_step()

perf.start_step("Materialize sorted rf_simples")
con.execute(f"CREATE TABLE rf_simples_sorted AS SELECT * FROM read_parquet('{BD}/rf_simples/**/*.parquet') ORDER BY cnpj_basico")
perf.end_step()

duckdb_register_domains(con)

# Pegamos as UFs direto do parquet para não materializar a maior tabela inteira
ufs = con.execute(f"SELECT DISTINCT uf FROM read_parquet('{BD}/rf_estabelecimentos/**/*.parquet') WHERE uf IS NOT NULL ORDER BY uf").fetchall()
ufs = [r[0] for r in ufs]

perf.start_step(f"Loop {len(ufs)} UFs (Sort-Merge Join)")
grand_total = 0
for uf in ufs:
    out_p = OUT_DIR / f"uf={uf}"
    out_p.mkdir(parents=True, exist_ok=True)
    out_f = (out_p / "data.parquet").as_posix()
    
    # Fazemos o JOIN usando a CTE que lê o parquet com filtro + tabelas pre-ordenadas
    sql = f'''
        COPY (
            WITH
                cte_est AS ({SQL_CTE_EST} FROM read_parquet('{BD}/rf_estabelecimentos/**/*.parquet') WHERE uf = '{uf}'),
                cte_emp AS ({SQL_CTE_EMP.replace("rf_empresas", "rf_empresas_sorted")}),
                cte_sim AS ({SQL_CTE_SIM.replace("rf_simples", "rf_simples_sorted")})
            {SQL_SELECT_FINAL}, '{uf}' AS uf
            {SQL_JOINS}
        ) TO '{out_f}' (FORMAT PARQUET)
    '''
    con.execute(sql)
    n = con.execute(f"SELECT count(*) FROM read_parquet('{out_f}')").fetchone()[0]
    grand_total += n
    print(f"     uf={uf:<3} rows={n:>10,}")

perf.end_step(grand_total)
con.close()
result_6 = perf.finish()

if DB_FILE.exists(): DB_FILE.unlink()

### Opção 7 - PySpark Local

In [ ]:
from pyspark.sql import SparkSession
import shutil

perf = PerfTracker("Option 7: PySpark Local (Spill-safe)")
perf.start()

if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

perf.start_step("Initialize Spark Session")
# Configuramos o Spark para usar o máximo possível da sua máquina e fazer spill pro disco se necessário
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("BenchmarkFornecedor360") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "12") \
    .getOrCreate()
perf.end_step()

perf.start_step("Register Views")
# Spark lê parquets lazy, não carrega tudo na RAM
spark.read.parquet(f"{BD}/rf_estabelecimentos").createOrReplaceTempView("rf_estabelecimentos")
spark.read.parquet(f"{BD}/rf_empresas").createOrReplaceTempView("rf_empresas")
spark.read.parquet(f"{BD}/rf_simples").createOrReplaceTempView("rf_simples")

domains = ["dom_naturezas", "dom_qualificacoes", "dom_cnaes", "dom_municipios", "dom_paises", "dom_motivos"]
paths = ["rf_naturezas", "rf_qualificacoes", "rf_cnaes", "rf_municipios", "rf_paises", "rf_motivos"]
for name, path in zip(domains, paths):
    spark.read.parquet(f"{BD}/{path}").createOrReplaceTempView(name)
perf.end_step()

perf.start_step("Execute Full Spark Pipeline (Write Partitioned)")

# O SQL do Spark é levemente diferente do DuckDB (to_date vs TRY_STRPTIME e concat vs ||)
SPARK_SQL = f'''
    WITH
    cte_est AS (
        SELECT
            concat(cnpj_basico, cnpj_ordem, cnpj_dv) AS cnpj_normalized,
            cnpj_basico, cnpj_ordem, cnpj_dv,
            CASE identificador_matriz_filial
                WHEN 1 THEN 'Matriz' WHEN 2 THEN 'Filial' ELSE 'Desconhecido'
            END AS tipo_estabelecimento,
            nome_fantasia, situacao_cadastral AS situacao_cadastral_cod,
            CASE situacao_cadastral
                WHEN '01' THEN 'Nula' WHEN '02' THEN 'Ativa' WHEN '03' THEN 'Suspensa'
                WHEN '04' THEN 'Inapta' WHEN '05' THEN 'Ativa Nao Regular'
                WHEN '08' THEN 'Baixada' WHEN '99' THEN 'Exclusao Logica' ELSE 'Desconhecida'
            END AS situacao_cadastral_desc,
            to_date(cast(data_situacao_cadastral as string), 'yyyyMMdd') AS data_situacao_cadastral,
            motivo_situacao_cadastral AS motivo_situacao_cadastral_cod,
            nome_cidade_exterior, pais AS pais_cod, uf,
            to_date(cast(data_inicio_atividade as string), 'yyyyMMdd') AS data_inicio_atividade,
            CASE WHEN cnae_fiscal_principal = '8888888' THEN NULL ELSE cnae_fiscal_principal END AS cnae_fiscal_principal_cod,
            cnae_fiscal_secundaria, tipo_logradouro, logradouro, numero, complemento, bairro, cep,
            municipio AS municipio_cod, situacao_especial,
            to_date(cast(data_situacao_especial as string), 'yyyyMMdd') AS data_situacao_especial
        FROM rf_estabelecimentos
    ),
    cte_emp AS (
        SELECT cnpj_basico, razao_social, natureza_juridica AS natureza_juridica_cod,
            qualificacao_responsavel AS qualificacao_responsavel_cod,
            cast(replace(capital_social, ',', '.') as decimal(15,2)) AS capital_social,
            CASE porte_empresa WHEN '01' THEN 'Micro Empresa' WHEN '03' THEN 'Empresa de Pequeno Porte' WHEN '05' THEN 'Demais' END AS porte_empresa,
            ente_federativo_responsavel
        FROM rf_empresas
    ),
    cte_sim AS (
        SELECT cnpj_basico,
            CASE opcao_simples WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE END AS opcao_simples,
            CASE opcao_mei WHEN 'S' THEN TRUE WHEN 'N' THEN FALSE END AS opcao_mei,
            to_date(cast(data_opcao_simples as string), 'yyyyMMdd') AS data_opcao_simples,
            CASE WHEN data_exclusao_simples != '00000000' THEN to_date(cast(data_exclusao_simples as string), 'yyyyMMdd') END AS data_exclusao_simples,
            CASE WHEN data_opcao_mei != '00000000' THEN to_date(cast(data_opcao_mei as string), 'yyyyMMdd') END AS data_opcao_mei,
            CASE WHEN data_exclusao_mei != '00000000' THEN to_date(cast(data_exclusao_mei as string), 'yyyyMMdd') END AS data_exclusao_mei
        FROM rf_simples
    )
    SELECT est.*,
        mot.motivo_descricao AS motivo_situacao_cadastral_desc,
        cnae.cnae_descricao AS cnae_fiscal_principal_desc,
        mun.municipio_descricao AS municipio_desc,
        pais.pais_descricao AS pais_desc,
        emp.razao_social, emp.natureza_juridica_cod, nat.natureza_descricao AS natureza_juridica_desc,
        emp.qualificacao_responsavel_cod, COALESCE(qual.qualificacao_descricao, 'Nao encontrada') AS qualificacao_responsavel_desc,
        emp.capital_social, emp.porte_empresa, emp.ente_federativo_responsavel,
        sim.opcao_simples, sim.data_opcao_simples, sim.data_exclusao_simples,
        sim.opcao_mei, sim.data_opcao_mei, sim.data_exclusao_mei,
        current_timestamp() AS _silver_loaded_at
    FROM cte_est est
    LEFT JOIN cte_emp emp ON est.cnpj_basico = emp.cnpj_basico
    LEFT JOIN cte_sim sim ON est.cnpj_basico = sim.cnpj_basico
    LEFT JOIN dom_naturezas nat ON emp.natureza_juridica_cod = nat.natureza_codigo
    LEFT JOIN dom_qualificacoes qual ON emp.qualificacao_responsavel_cod = qual.qualificacao_codigo
    LEFT JOIN dom_cnaes cnae ON est.cnae_fiscal_principal_cod = cnae.cnae_codigo
    LEFT JOIN dom_municipios mun ON est.municipio_cod = mun.municipio_codigo
    LEFT JOIN dom_paises pais ON est.pais_cod = pais.pais_codigo
    LEFT JOIN dom_motivos mot ON est.motivo_situacao_cadastral_cod = mot.motivo_codigo
'''

# Roda o job e escreve direto no disco particionado
df = spark.sql(SPARK_SQL)
df.write.mode("overwrite").partitionBy("uf").parquet(str(OUT_DIR))

# Conta as linhas escritas
total_rows = spark.read.parquet(str(OUT_DIR)).count()
perf.end_step(total_rows)

spark.stop()
result_7 = perf.finish()

## Results Comparison

In [ ]:
# Collect all results that were run
results = {}
for var_name in ['result_1','result_2','result_3','result_4','result_5', 'result_6', 'result_7']:
    if var_name in dir():
        r = eval(var_name)
        results[r['name']] = r

if results:
    print(f"{'Option':<50} {'Time':>8} {'Rows':>15} {'Output':>10}")
    print("-" * 90)
    for name, r in sorted(results.items()):
        print(f"{name:<50} {r['total_seconds']:>6.1f}s {r['total_rows']:>15,} {r['output_mb']:>8.1f}MB")
else:
    print("No results yet. Run at least one option first.")